In [ ]:
import copy
import pandas as pd
import torch

from pivotal_tokens.extractor import THINKING_START_TOKEN, THINKING_END_TOKEN, TRUNCATED_TAG
from pivotal_tokens.hf.loading import load_model, load_tokenizer
import plotly.express as px


torch.set_grad_enabled(False)


QWEN3_1_7B_MODEL_ID = "Qwen/Qwen3-1.7B"
DEVICE = "cuda"


SYSTEM_PROMPT = (
    'Answer the following question directly, strictly without chain-of-thought after "</think>". '
    'The reasoning trace may be truncated; if so, it will end with "<truncated>" placed right before '
    'the closing "</think>". In that case, use the partial reasoning to make your best guess at the answer. '
    'Keep the answer short (e.g., "yes" or "no" for binary questions, a person\'s full name if the question '
    'expects a person, a country name if it asks about a country, etc.). '
    'Output the answer strictly after the prefix "Answer: `<answer>`" with no extra text.'
)

REASONING_TRACE = """<think>
Okay, let's see. The question is asking in which province the city is located where Ed Wubbe was born. First, I need to figure out where Ed Wubbe is from. I remember that Ed Wubbe is a Dutch politician, right? He's been involved in the Dutch government, maybe as a minister.

I think he was born in the Netherlands. But the question is about the province. So I need to recall or find out the specific province. Let me think... I recall that Ed Wubbe was born in Hilversum. Hilversum is a city in the province of North Holland. Wait, is that correct? Let me double-check. Yes, Hilversum is indeed in North Holland. So the answer should be North Holland. But wait, maybe I'm mixing up the provinces. Let me make sure. North Holland is one of the provinces in the Netherlands. Yes, Hilversum is a city in that province. So the answer is North Holland.
"""

CONTEXT = """<|im_start|>system
Answer the following question directly, strictly without chain-of-thought after "</think>". The reasoning trace may be truncated; if so, it will end with "<truncated>" placed right before the closing "</think>". In that case, use the partial reasoning to make your best guess at the answer. Keep the answer short (e.g., "yes" or "no" for binary questions, a person's full name if the question expects a person, a country name if it asks about a country, etc.). Output the answer strictly after the prefix "Answer: `<answer>`" with no extra text.<|im_end|>
<|im_start|>user
In which province is the city located that is the birthplace of Ed Wubbe?<|im_end|>
<|im_start|>assistant
"""

EXPECTED_ANSWER = "North Holland"

ANSWER_SUFFIX = f"{TRUNCATED_TAG}\n{THINKING_END_TOKEN}\nAnswer: `{EXPECTED_ANSWER}`"

In [ ]:
model = load_model(model_id=QWEN3_1_7B_MODEL_ID, device=DEVICE, use_flash_attention=False)
tokenizer = load_tokenizer(model_id=QWEN3_1_7B_MODEL_ID)

## Option 1: one-shot NNLL calculation

In [ ]:
n_reasoning_trace_tokens = 2

trace_tok_ids = tokenizer.encode(REASONING_TRACE, add_special_tokens=False)
prefix_text = tokenizer.decode(trace_tok_ids[:n_reasoning_trace_tokens], skip_special_tokens=False)

print(f"Prefix ({n_reasoning_trace_tokens} tokens):")
print(repr(prefix_text))

In [ ]:
full_prefix = CONTEXT + prefix_text
full_prefix_tok = tokenizer(full_prefix, return_tensors="pt", return_attention_mask=False).to(model.device)

In [ ]:
conditional_context = full_prefix + ANSWER_SUFFIX
conditional_tok = tokenizer(conditional_context, return_tensors="pt").to(model.device)
n_prefix_tokens = full_prefix_tok.input_ids.shape[1]
suffix_ids = conditional_tok.input_ids[:, n_prefix_tokens:]

In [ ]:
out_prefix = model(input_ids=full_prefix_tok.input_ids, use_cache=True)

# Deep-copy KV cache to prevent DynamicCache in-place mutation during suffix forward pass
out_suffix = model(input_ids=suffix_ids[:, :-1], use_cache=False, past_key_values=copy.deepcopy(out_prefix.past_key_values))

logits_for_loss = torch.cat([out_prefix.logits[:, -1, :].unsqueeze(1), out_suffix.logits], dim=1)

In [ ]:
n_ground_truth_tokens = len(tokenizer.encode(EXPECTED_ANSWER))
target = suffix_ids.reshape(-1).clone()
target[:-(n_ground_truth_tokens + 1)] = -100
target[-1] = -100

In [ ]:
nnll = torch.nn.functional.cross_entropy(
    input=logits_for_loss.reshape(-1, logits_for_loss.size(-1)),
    target=target,
    reduction="none",
)

In [ ]:
token_ids = suffix_ids.tolist()[0]
for token_id, score in zip(token_ids, nnll.tolist()):
    if token_id == -100:
        continue
    token = tokenizer.decode([token_id])
    print(f"{repr(token):<10} = {score:.3f}")

In [ ]:
oneshot_kv = out_prefix.past_key_values
oneshot_logits = logits_for_loss.reshape(-1, logits_for_loss.size(-1))

## Option 2: iterative NNLL calculation

In [ ]:
completion_tok = tokenizer(REASONING_TRACE, return_tensors="pt").to(model.device)
completion_len = completion_tok.input_ids.shape[1]

full_text = CONTEXT + REASONING_TRACE
full_tok = tokenizer(full_text, return_tensors="pt", return_attention_mask=False).to(model.device)

# Indices start from the <thinking> token in order to get nnll of the "no thinking" scenario.
# Specifically, the case "... <thinking> </thinking> ..."
completion_start_idx = full_tok.input_ids.shape[1] - completion_len + 1
completion_end_idx = full_tok.input_ids.shape[1]
indices = list(range(completion_start_idx, completion_end_idx))

# indices = indices[:1]
indices = indices[:2]

In [ ]:
n_ground_truth_tokens = len(tokenizer.encode(EXPECTED_ANSWER))
n_ground_truth_tokens

In [ ]:
past_key_values = None
prev_t = 0
last_prefix_logits = None

### Iter 1

In [ ]:
t = indices[0]

In [ ]:
new_tokens = full_tok.input_ids[:, prev_t:t]
print(tokenizer.decode(new_tokens[0].tolist()))

out_prefix = model(input_ids=new_tokens,
                    use_cache=True)

past_key_values = out_prefix.past_key_values
prev_t = t

In [ ]:
prefix_ids = full_tok.input_ids[:, :t]
prefix_str = tokenizer.decode(prefix_ids[0], skip_special_tokens=False)

print(prefix_str)

In [ ]:
conditional_context = prefix_str + ANSWER_SUFFIX
conditional_context_tok = tokenizer(conditional_context, return_tensors="pt").to(model.device)

print(conditional_context)

In [ ]:
suffix_ids = conditional_context_tok.input_ids[:, t:]
suffix_input = suffix_ids[:, :-1]

print(tokenizer.decode(suffix_ids[0].tolist()))

In [ ]:
out_suffix = model(input_ids=suffix_input, use_cache=False, past_key_values=copy.deepcopy(past_key_values))
past_key_values = out_prefix.past_key_values

In [ ]:
prev_last_prefix_logits = last_prefix_logits
last_prefix_logits = out_prefix.logits[:, -1, :]

print("last_prefix_logits shape", last_prefix_logits.shape)

In [ ]:
logits_for_loss = torch.cat([last_prefix_logits.unsqueeze(1),
                            out_suffix.logits],
                            dim=1)

print(logits_for_loss.shape)

In [ ]:
target = suffix_ids.reshape(-1).clone()
target[:-(n_ground_truth_tokens + 1)] = -100
target[-1] = -100

print(target)

In [ ]:
nnll = torch.nn.functional.cross_entropy(
    input=logits_for_loss.reshape(-1, logits_for_loss.size(-1)),
    target=target,
    reduction="none",
)

token_ids = suffix_ids[0].tolist()
for token_id, score in zip(token_ids, nnll.tolist()):
    if token_id == -100:
        continue
    token = tokenizer.decode([token_id])
    print(f"{repr(token):<10} = {score:.10f}")

### Iter 2

In [ ]:
t = indices[1]

In [ ]:
new_tokens = full_tok.input_ids[:, prev_t:t]
print(repr(tokenizer.decode(new_tokens[0].tolist())))

In [ ]:
past_length = past_key_values[0][0].shape[2]
seq_len = new_tokens.shape[1]
total_len = past_length + seq_len

position_ids = torch.arange(past_length, total_len, device=model.device).unsqueeze(0)
attention_mask = torch.ones(1, total_len, device=model.device, dtype=torch.long)

out_prefix = model(input_ids=new_tokens,
                    past_key_values=past_key_values,
                    position_ids=position_ids,
                    attention_mask=attention_mask,
                    use_cache=True)

past_key_values = out_prefix.past_key_values
prev_t = t

In [ ]:
prefix_ids = full_tok.input_ids[:, :t]
prefix_str = tokenizer.decode(prefix_ids[0], skip_special_tokens=False)

print(prefix_str)

In [ ]:
conditional_context = prefix_str + ANSWER_SUFFIX
conditional_context_tok = tokenizer(conditional_context, return_tensors="pt").to(model.device)

print(conditional_context)

In [ ]:
suffix_ids = conditional_context_tok.input_ids[:, t:]
suffix_input = suffix_ids[:, :-1]

print(tokenizer.decode(suffix_ids[0].tolist()))

In [ ]:
out_suffix = model(input_ids=suffix_input, use_cache=False, past_key_values=copy.deepcopy(past_key_values))
past_key_values = out_prefix.past_key_values

In [ ]:
prev_last_prefix_logits = last_prefix_logits
last_prefix_logits = out_prefix.logits[:, -1, :]

print("last_prefix_logits shape", last_prefix_logits.shape)

In [ ]:
logits_for_loss = torch.cat([last_prefix_logits.unsqueeze(1),
                            out_suffix.logits],
                            dim=1)

print(logits_for_loss.shape)

In [ ]:
target = suffix_ids.reshape(-1).clone()
target[:-(n_ground_truth_tokens + 1)] = -100
target[-1] = -100

print(target)

In [ ]:
nnll = torch.nn.functional.cross_entropy(
    input=logits_for_loss.reshape(-1, logits_for_loss.size(-1)),
    # target=target,
    target=suffix_ids.reshape(-1),
    reduction="none",
)

token_ids = suffix_ids[0].tolist()
for token_id, score in zip(token_ids, nnll.tolist()):
    if token_id == -100:
        continue
    token = tokenizer.decode([token_id])
    print(f"{repr(token):<10} = {score:.10f}")

In [ ]:
iter2_kv = out_prefix.past_key_values
iter2_last_prefix_logits = logits_for_loss.reshape(-1, logits_for_loss.size(-1))

# Comparison

In [ ]:
torch.allclose(oneshot_kv[0][0], iter2_kv[0][0], atol=1e-5)

In [ ]:
torch.allclose(oneshot_logits, iter2_last_prefix_logits, atol=1e-5)